Construction of NicheNet’s ligand-target model
================

This vignette shows how ligand-target prior regulatory potential scores
are inferred in the NicheNet framework. You can use the procedure shown
here to develop your own model with inclusion of context-specific
networks or removal of noisy irrelevant data sources. The networks at
the basis of NicheNet can be downloaded from Zenodo
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.15592385.svg)](https://doi.org/10.5281/zenodo.15592385).

# Background information about NicheNet’s prior ligand-target model

The prior model at the basis of NicheNet denotes how strongly existing
knowledge supports that a ligand may regulate the expression of a target
gene. To calculate this ligand-target regulatory potential, we
integrated biological knowledge about ligand-to-target signaling paths
as follows.

First, we collected multiple complementary data sources covering
ligand-receptor, signal transduction (e.g., protein-protein and
kinase-substrate interactions) and gene regulatory interactions (e.g.,
inferred from ChIP-seq and motifs). 

Secondly, we integrated these individual data sources into two weighted
networks: 1) a ligand-signaling network, which contains protein-protein
interactions covering the signaling paths from ligands to downstream
transcriptional regulators; and 2) a gene regulatory network, which
contains gene regulatory interactions between transcriptional regulators
and target genes. To let informative data sources contribute more to the
final model, we weighted each data source during integration. These data
source weights were automatically determined via model-based parameter
optimization to improve the accuracy of ligand-target predictions. In this tutorial, we will show how
to construct models with unoptimized data source weigths as well.

Finally, we combined the ligand-signaling and gene regulatory network to
calculate a regulatory potential score between all pairs of ligands and
target genes. A ligand-target pair receives a high regulatory potential
if the regulators of the target gene are lying downstream of the
signaling network of the ligand. To calculate this, we used network
propagation methods on the integrated networks to propagate the signal
starting from a ligand, flowing through receptors, signaling proteins,
transcriptional regulators, and ultimately ending at target genes.

A graphical summary of this procedure is visualized here below:

![](images/workflow_model_construction.png)

# Construct a ligand-target model from all collected ligand-receptor, signaling and gene regulatory network data sources

Import the required packages. 

In [ ]:
from nichenetpy.utils import read_csv_cols, read_csv_rows
from nichenetpy.model_construction import (
    construct_weighted_networks,
    construct_ligand_target_matrix,
    apply_hub_correction
)

from itertools import chain, repeat

import os
import requests
import pandas as pd
import session_info

Dowload the networks we will use to construct the model. 

In [2]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv",
    "optimized_source_weights.csv",
    "annotation_data_sources.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

## Construct NicheNet’s ligand-target model from unoptimized data source weights

Construct the weighted integrated ligand-signaling and gene regulatory
network. In this first example, we give every data source the same
weight. For the hyperparameters
of the model (hub correction factors and damping factor), we will use
the optimized values.

The ligand-signaling network hub correction factor and gene regulatory
network hub correction factor were defined as hyperparameter of the
model to mitigate the potential negative influence of over-dominant hubs
on the final model. The damping factor hyperparameter is the main
parameter of the Personalized PageRank algorithm, which we used as
network propagation algorithm to link ligands to downstream regulators.

In [4]:
source_weights = dict(zip(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])), repeat(1)))
weighted_networks = construct_weighted_networks(
    lr_network,
    sig_network,
    gr_network,
    source_weights
)
weighted_networks["lr_sig"] = apply_hub_correction(weighted_networks["lr_sig"], hub=0.115)
weighted_networks["gr"] = apply_hub_correction(weighted_networks["gr"], hub=0.0803)

Infer ligand-target regulatory potential scores based on the weighted integrated networks

In [5]:
ligands = [["TNF"], ["TNF", "IL6"]]
mat, row_names, col_names = construct_ligand_target_matrix(
    weighted_networks,
    lr_network,
    ligands,
    damping_factor=0.789,
    ltf_cutoff=0.926
)

In [6]:
pd.DataFrame(mat, index=row_names, columns=col_names)

,TNF,TNF-IL6
A-GAMMA3'E,0.000000,0.000000
A1BG,0.010947,0.012134
A1BG-AS1,0.005083,0.005172
A1CF,0.012494,0.013481
A2M,0.302233,0.244572
...,...,...
ZYG11A,0.021182,0.023158
ZYG11B,0.024555,0.026096
ZYX,0.030305,0.034638
ZZEF1,0.034764,0.038650


## Construct NicheNet’s ligand-target model from optimized data source weights

Now, we will demonstrate how you can make an alternative model with the
optimized data source weights. 

In [7]:
optimized_source_weights = tuple(zip(*read_csv_rows("./tutorial_files/model_construction/human/optimized_source_weights.csv")[1]))
optimized_source_weights = dict(zip(optimized_source_weights[0], [float(e) for e in optimized_source_weights[1]]))
weighted_networks = construct_weighted_networks(
    lr_network,
    sig_network,
    gr_network,
    optimized_source_weights
)
weighted_networks["lr_sig"] = apply_hub_correction(weighted_networks["lr_sig"], hub=0.115)
weighted_networks["gr"] = apply_hub_correction(weighted_networks["gr"], hub=0.0803)

In [8]:
ligands = [["TNF"]]
mat, row_names, col_names = construct_ligand_target_matrix(
    weighted_networks,
    lr_network,
    ligands,
    damping_factor=0.789,
    ltf_cutoff=0.926
)

In [10]:
pd.DataFrame(mat, index=row_names, columns=col_names)

,TNF
A-GAMMA3'E,0.000000
A1BG,0.002488
A1BG-AS1,0.000723
A1CF,0.003257
A2M,0.110323
...,...
ZYG11A,0.003894
ZYG11B,0.004144
ZYX,0.006908
ZZEF1,0.005442


# Change the data sources at the basis of the NicheNet ligand-target model

### Keep only specific data sources of interest

Now, we will demonstrate how you can decide which data sources to use in
the model you want to create. Let’s say for this example, that you are
interested in making a model that only consists of literature-derived
ligand-receptor interactions, signaling and gene regulatory interactions
from comprehensive databases and gene regulatory interactions inferred
from ChIP-seq. 

In [11]:
annotations = pd.DataFrame(read_csv_cols("./tutorial_files/model_construction/human/annotation_data_sources.csv"))
data_sources_to_keep = set(annotations[[e in ["literature", "comprehensive_db", "ChIP"] for e in annotations["type_db"]]]["source"])
new_source_weights = dict((key, val) for key, val in source_weights.items() if key in data_sources_to_keep)
new_lr_network = lr_network[[e in data_sources_to_keep for e in lr_network["source"]]]
new_sig_network = sig_network[[e in data_sources_to_keep for e in sig_network["source"]]]
new_gr_network = gr_network[[e in data_sources_to_keep for e in gr_network["source"]]]

In [12]:
weighted_networks = construct_weighted_networks(
    new_lr_network,
    new_sig_network,
    new_gr_network,
    new_source_weights
)
weighted_networks["lr_sig"] = apply_hub_correction(weighted_networks["lr_sig"], hub=0.115)
weighted_networks["gr"] = apply_hub_correction(weighted_networks["gr"], hub=0.0803)
ligands = [["TNF"]]
mat, row_names, col_names = construct_ligand_target_matrix(
    weighted_networks,
    new_lr_network,
    ligands,
    damping_factor=0.789,
    ltf_cutoff=0.926
)

In [13]:
pd.DataFrame(mat, index=row_names, columns=col_names)

,TNF
A1BG,0.008213
A1BG-AS1,0.004989
A1CF,0.008266
A2M,0.018543
A2M-AS1,0.002882
...,...
ZYG11A,0.017568
ZYG11B,0.018650
ZYX,0.018481
ZZEF1,0.028591


In [14]:
session_info.show()